# SplatStream Lab — Reproducible CUDA Evaluation

This notebook reproduces the real-scene validation path for SplatStream Lab using **Mip-NeRF 360 / Bonsai** and a pinned `gsplat` revision.

The first stage records the software/hardware environment, trains a 7,000-step 3D Gaussian Splatting baseline, exports the resulting checkpoint to PLY, and executes the project's portable compression sanity-check pipeline.

The second stage reuses the trained checkpoint to run a **full-scene held-out CUDA rate-distortion sweep** across pruning and quantization operating points.

**Runtime requirement:** select a GPU runtime before execution.


## 1. Train and validate the 7K Bonsai baseline


In [ ]:
from pathlib import Path
import urllib.request

runner = Path('/content/run_bonsai_cuda.py')
url = 'https://raw.githubusercontent.com/reusahn/splatstream-lab/main/tools/run_bonsai_cuda.py'
urllib.request.urlretrieve(url, runner)
print('Runner:', runner)

%run /content/run_bonsai_cuda.py


## 2. Full-scene CUDA rate-distortion sweep

This stage uses the checkpoint produced above. It applies the same view-independent importance heuristic to the full scene, creates four pruning/quantization operating points, and evaluates every point on the same held-out Bonsai cameras with the full `gsplat` rasterizer.

The evaluator keeps one `gsplat` Runner, dataset, CUDA stack, and metric networks alive for the entire sweep. This avoids repeated process startup and repeated JIT/metric initialization between operating points.

The sweep records PSNR, SSIM, LPIPS, render time, exact experimental payload bytes, compression ratios, and validation renders. The 8-bit payload uses per-channel min/max side information and zlib as a generic entropy backend. Dequantized tensors are used only for rasterization, not as compressed-size measurements.


In [ ]:
from pathlib import Path
import urllib.request

rd_runner = Path('/content/full_cuda_rate_distortion_v2.py')
rd_url = 'https://raw.githubusercontent.com/reusahn/splatstream-lab/main/tools/full_cuda_rate_distortion_v2.py'
urllib.request.urlretrieve(rd_url, rd_runner)
print('Rate-distortion runner:', rd_runner)

%run /content/full_cuda_rate_distortion_v2.py


## Outputs

The baseline stage produces **`splatstream_bonsai_evidence.zip`**, containing the environment manifest, training log, exported metric JSON files, checkpoint/PLY inventories, and portable real-PLY sanity-check outputs.

The full CUDA sweep produces **`splatstream_full_cuda_rd_v2.zip`**, containing rate-distortion CSV/JSON, method notes, environment metadata, plots, and the validation outputs generated by the single-process evaluator.

Measured values should be reported together with the GPU model, pinned commit IDs, dataset source, evaluation split, and payload definition.
